In [1]:
import torch
import torch.nn as nn
import timm
from PIL import Image
from torchvision import transforms

In [2]:
class SpatialCNN(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()

        self.backbone = timm.create_model(
            "efficientnet_b0",
            pretrained=False,
            num_classes=0,
            global_pool="avg"
        )

        in_features = self.backbone.num_features

        self.proj = nn.Sequential(
            nn.Linear(in_features, out_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5)
        )

        self.out_dim = out_dim

    def forward(self, x):
        x = self.backbone(x)
        x = self.proj(x)
        return x

In [3]:
class FrequencyCNN(nn.Module):
    def __init__(self, out_dim=64, low_cut=0.02, high_cut=0.60):
        super().__init__()
        self.low_cut = low_cut
        self.high_cut = high_cut

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 96, kernel_size=3, padding=1),
            nn.BatchNorm2d(96),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(96, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, out_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5)
        )

        self.out_dim = out_dim

    def forward(self, x):
        fft = torch.fft.rfft2(x, dim=(-2, -1))
        amp = torch.log1p(torch.abs(fft))

        h, w = amp.shape[-2], amp.shape[-1]

        yy = torch.linspace(-1.0, 1.0, steps=h, device=amp.device)
        xx = torch.linspace(0.0, 1.0, steps=w, device=amp.device)
        yy, xx = torch.meshgrid(yy, xx, indexing="ij")

        radius = torch.sqrt(xx ** 2 + yy ** 2)
        radius = radius / (radius.max() + 1e-6)

        mask = ((radius >= self.low_cut) & (radius <= self.high_cut)).float()
        amp = amp * mask.view(1, 1, h, w)

        amp = amp - amp.mean(dim=(-2, -1), keepdim=True)
        amp = amp / (amp.std(dim=(-2, -1), keepdim=True) + 1e-6)

        x = self.features(amp)
        x = self.proj(x)
        return x

In [4]:
class FreqNet(nn.Module):
    def __init__(self, num_classes=2, spatial_dim=128, freq_dim=64):
        super().__init__()

        self.spatial_branch = SpatialCNN(out_dim=spatial_dim)
        self.frequency_branch = FrequencyCNN(out_dim=freq_dim)

        fusion_dim = spatial_dim + freq_dim

        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        spatial_feat = self.spatial_branch(x)
        freq_feat = self.frequency_branch(x)
        fused = torch.cat([spatial_feat, freq_feat], dim=1)
        return self.classifier(fused)

In [5]:
val_transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.5] * 3)
])

In [6]:
#Путь для теста на конкретном изображении.

model_path = "best_model.pth"
image_path = 'images/0.jpg'

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [8]:
model = FreqNet(num_classes=2).to(device)

checkpoint = torch.load(model_path, map_location=device)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model.eval()

FreqNet(
  (spatial_branch): SpatialCNN(
    (backbone): EfficientNet(
      (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn1): BatchNormAct2d(
        32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
        (drop): Identity()
        (act): SiLU(inplace=True)
      )
      (blocks): Sequential(
        (0): Sequential(
          (0): DepthwiseSeparableConv(
            (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (bn1): BatchNormAct2d(
              32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
              (drop): Identity()
              (act): SiLU(inplace=True)
            )
            (aa): Identity()
            (se): SqueezeExcite(
              (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
              (act1): SiLU(inplace=True)
              (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1

In [9]:
image = Image.open(image_path).convert("RGB")
x = val_transform(image).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(x)
    probs = torch.softmax(logits, dim=1)
    pred = torch.argmax(probs, dim=1).item()

print("Predicted class:", pred)
print("Probabilities:", probs.cpu().numpy())

Predicted class: 1
Probabilities: [[0.06864695 0.9313531 ]]
